In [5]:
import numpy as np
from scipy.integrate import solve_ivp
import plotly.graph_objects as go
import plotly.io as pio


class RlcSolenoidCircuit:
    """
    Circuit model:

        2L * d2i/dt2 + 2R * di/dt + (1/C) * i = dv/dt

    Generic Source:
        v(t) = user-defined function
        dv/dt can be provided explicitly for better accuracy.
    """

    def __init__(
        self,
        v_function,         # Function for voltage source v(t)
        dv_dt_function,     # Function for dv/dt, can be None
        R,
        C,
        N,
        r,
        l,
        dt,
        t_end,
        plot_width=1400,
        plot_height=550,
    ):
        self.v_function = v_function
        self.dv_dt_function = dv_dt_function

        self.R = float(R)
        self.C = float(C)
        self.N = float(N)
        self.r = float(r)
        self.l = float(l)

        self.dt = float(dt)
        self.t_end = float(t_end)

        self.plot_width = int(plot_width)
        self.plot_height = int(plot_height)

        self.L = None
        self.t_vec = None
        self.i_vec = None

    # --------------------------------------------------------
    # Inductance (Wheeler approximation adapted to SI)
    # --------------------------------------------------------
    def compute_inductance(self):
        numerator = 1e-6 * (self.N * self.N * self.r * self.r)
        denominator = 9.0 * self.r + 10.0 * self.l
        self.L = numerator / denominator
        return self.L

    # --------------------------------------------------------
    # Voltage derivative (dv/dt)
    # Priority: user-provided dv_dt_function
    # Fallback: numerical derivative with central difference
    # --------------------------------------------------------
    def dv_dt(self, t):
        if self.dv_dt_function is not None:
            return float(self.dv_dt_function(t))

        # Numerical fallback (central difference when possible)
        eps = 0.1 * self.dt
        if eps <= 0.0:
            eps = 1e-9

        if t >= eps:
            v_plus = self.v_function(t + eps)
            v_minus = self.v_function(t - eps)
            return float((v_plus - v_minus) / (2.0 * eps))

        v0 = self.v_function(t)
        v1 = self.v_function(t + eps)
        return float((v1 - v0) / eps)

    # --------------------------------------------------------
    # ODE system
    # --------------------------------------------------------
    def ode_system(self, t, x):
        i = x[0]
        di_dt = x[1]

        d2i_dt2 = (
            self.dv_dt(t)
            - 2.0 * self.R * di_dt
            - (1.0 / self.C) * i
        ) / (2.0 * self.L)

        return [di_dt, d2i_dt2]

    # --------------------------------------------------------
    # Time grid (robust: integer number of steps)
    # --------------------------------------------------------
    def build_time_grid(self):
        n_steps = int(np.floor(self.t_end / self.dt)) + 1
        t_last = self.dt * float(n_steps - 1)
        t_eval = np.linspace(0.0, t_last, n_steps)
        return t_eval

    # --------------------------------------------------------
    # Simulation
    # --------------------------------------------------------
    def simulate(self, i0=0.0, di_dt0=0.0):
        if self.L is None:
            self.compute_inductance()

        t_eval = self.build_time_grid()

        x0 = [float(i0), float(di_dt0)]

        t0 = float(t_eval[0])
        tf = float(t_eval[-1])

        sol = solve_ivp(
            fun=self.ode_system,
            t_span=(t0, tf),
            y0=x0,
            t_eval=t_eval,
            method="RK45",
            rtol=1e-8,
            atol=1e-10
        )

        self.t_vec = sol.t
        self.i_vec = sol.y[0]

        return self.t_vec, self.i_vec

    # --------------------------------------------------------
    # Plot (Plotly for Colab)
    # --------------------------------------------------------
    def plot(self):
        if self.t_vec is None or self.i_vec is None:
            raise RuntimeError("Simulation not executed yet.")

        pio.renderers.default = "colab"

        fig = go.Figure()

        fig.add_trace(
            go.Scatter(
                x=self.t_vec,
                y=self.i_vec,
                mode="lines",
                name="i(t)"
            )
        )

        fig.update_layout(
            width=self.plot_width,
            height=self.plot_height,
            template="plotly_white",
            xaxis=dict(
                title="t [s]",
                autorange=True,
                fixedrange=False
            ),
            yaxis=dict(
                title="i(t) [A]",
                autorange=True,
                fixedrange=False
            )
        )


        fig.show()


# ============================================================
# Main execution (Colab)
# ============================================================
if __name__ == "__main__":

    # ---- Source definition (transient-friendly) ----
    # v(t) = (1 - exp(-t/tau)) * cos(w1*t), for t >= 0
    # This ramp excites the natural mode without singular dv/dt at t=0.

    Vp = 1.0
    w1 = 377.0
    tau = 5e-9

    def v(t):
        if t < 0.0:
            return 0.0
        return (1.0 - np.exp(-t / tau)) * Vp * np.cos(w1 * t)

    def dvdt(t):
        if t < 0.0:
            return 0.0
        a = 1.0 - np.exp(-t / tau)
        da_dt = (1.0 / tau) * np.exp(-t / tau)
        return da_dt * Vp * np.cos(w1 * t) + a * (-Vp * w1 * np.sin(w1 * t))

    # Instantiate the circuit
    circuit = RlcSolenoidCircuit(
        v_function=v,
        dv_dt_function=dvdt,   # <-- key change: provide analytic dv/dt
        R=0.1,
        C=25e-6,
        N=5.0,
        r=0.02,
        l=0.04,
        dt=1e-7,
        t_end=0.008,
        plot_width=1400,
        plot_height=550,
    )

    circuit.compute_inductance()

    # Initial conditions
    circuit.simulate(i0=0.0, di_dt0=0.0)

    circuit.plot()


Output hidden; open in https://colab.research.google.com to view.